# 10 — Knowledge map: Bayesian Knowledge Tracing on ASSISTments

The knowledge map shows, per concept, how well a student knows it. Two signals feed it: **how the student answered**
(this notebook) and **how the memory forecast says the concept is decaying** (notebook 08). Bayesian Knowledge Tracing
(BKT; Corbett & Anderson 1995) is the model for the first: a concept is either known or not, each answer is right with
probability 1 - slip if it is known and with probability guess if not, and after every answer the student may learn it.
The state is the probability that the concept is known. Choice questions are easier to guess than written ones, so the
two kinds have their own guess parameter (`ml_pipeline/bkt.py`).

**Data.** ASSISTments 2009-2010 skill builder (Feng, Heffernan & Koedinger 2009; mirror `Atomi/ASSISTments2009`; see
`datasets/README.md`): 4,148 students, 274,331 responses, 139 skills, each response with its answer type. Students are
split 80/20 by student with `config.RANDOM_SEED`, so the test students are never seen when fitting. The sequence of
one student on one skill is one BKT sequence.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from ml_pipeline import config, bkt

raw = pd.read_parquet(config.REPO_ROOT / 'datasets' / 'raw' / 'ASSISTments' / 'assist2009.parquet')
print(f"{len(raw)} students, {sum(len(x) for x in raw.grades)} responses, "
      f"{len({s for x in raw.skill_ids for s in x})} skills")

def sequences_of(row):
    by = {}
    for skill, grade, atype in zip(row.skill_ids, row.grades, row.answer_types):
        if atype == 'open_response':      # 8 answers, graded by hand: not a right/wrong event
            continue
        by.setdefault(skill, []).append((grade == '1', bkt.kind_of(atype)))
    return list(by.values())

rng = np.random.default_rng(config.RANDOM_SEED)
order = rng.permutation(len(raw))
cut = int(0.8 * len(raw))
train_students, test_students = raw.iloc[order[:cut]], raw.iloc[order[cut:]]
train = [s for r in train_students.itertuples() for s in sequences_of(r)]
test = [s for r in test_students.itertuples() for s in sequences_of(r)]
for name, seqs, students in [('train', train, train_students), ('test', test, test_students)]:
    print(f"{name}: {len(students)} students, {len(seqs)} (student, skill) sequences, "
          f"{sum(len(s) for s in seqs)} answers")
kinds = pd.Series([k for s in train for _, k in s]).value_counts()
print('train answers by kind:', kinds.to_dict())

4148 students, 274331 responses, 139 skills
train: 3318 students, 33502 (student, skill) sequences, 227069 answers
test: 830 students, 7274 (student, skill) sequences, 47254 answers
train answers by kind: {'written': 202088, 'choice': 24981}


## Fit

`bkt.fit` maximises the likelihood of the training sequences with L-BFGS-B, with each parameter bounded to a sensible
range. The likelihood is plain Python, so the first cell below times one evaluation: if a fit would take more than
about three minutes the training students are subsampled (1,500), otherwise all of them are used.

In [2]:
t0 = time.time()
bkt.neg_log_likelihood(train, bkt.Params(0.3, 0.15, 0.1, 0.25, 0.1))
one_eval = time.time() - t0
# L-BFGS-B with finite differences costs 6 evaluations per iteration; 100 iterations is a generous ceiling.
estimate = one_eval * 6 * 100
print(f"one likelihood evaluation: {one_eval:.2f}s; worst-case fit about {estimate / 60:.1f} min")

fit_students = train_students
if estimate > 180:
    fit_students = train_students.sample(1500, random_state=config.RANDOM_SEED)
    print('subsampled to 1,500 training students')
fit_seqs = [s for r in fit_students.itertuples() for s in sequences_of(r)]

t0 = time.time()
params = bkt.fit(fit_seqs)
elapsed = time.time() - t0
print(f"fitted on {len(fit_students)} students, {len(fit_seqs)} sequences, "
      f"{sum(len(s) for s in fit_seqs)} answers in {elapsed:.0f}s")
pd.Series(params.__dict__, name='fitted').to_frame()

one likelihood evaluation: 0.11s; worst-case fit about 1.1 min


fitted on 3318 students, 33502 sequences, 227069 answers in 21s


,fitted
init,0.740669
learn,0.055021
slip,0.204407
guess_choice,0.500000
guess_written,0.261751


## Does it predict the next answer?

For every answer of the test students, the model predicts it from the state **before** that answer, then updates.
BKT predicts with `p_correct(p_known, kind)`. The baseline is what a plain tally would say: the student's running
accuracy on that skill with a 0.6 prior, `(right so far + 0.6) / (answered so far + 1)`. AUC is computed over all test
answers; the baseline ignores the answer kind and the learning curve.

In [3]:
rows = []   # one per test answer: bkt prediction, baseline prediction, outcome, p_known before
for seq in test:
    p, right = params.init, 0
    for i, (correct, kind) in enumerate(seq):
        rows.append((bkt.p_correct(p, kind, params), (right + 0.6) / (i + 1), int(correct), p, kind))
        p = bkt.update(p, correct, kind, params)
        right += int(correct)
pred = pd.DataFrame(rows, columns=['bkt', 'baseline', 'correct', 'p_known', 'kind'])
auc = roc_auc_score(pred.correct, pred.bkt)
baseline_auc = roc_auc_score(pred.correct, pred.baseline)
print(f"{len(pred)} test answers, {pred.correct.mean():.3f} right")
print(f"AUC  BKT {auc:.4f}   running-accuracy baseline {baseline_auc:.4f}")
for kind, g in pred.groupby('kind'):
    print(f"  {kind:8s} n={len(g):6d}  BKT {roc_auc_score(g.correct, g.bkt):.4f}  baseline {roc_auc_score(g.correct, g.baseline):.4f}")

47254 test answers, 0.664 right
AUC  BKT 0.6899   running-accuracy baseline 0.6950
  choice   n=  4980  BKT 0.6895  baseline 0.6995
  written  n= 42274  BKT 0.6887  baseline 0.6938


## Thresholds: strong, shaky, weak

The map needs three bands. A concept is *strong* when `p_known` is at least the upper cut, *weak* below the lower cut,
and *shaky* between. For each candidate pair the test answers are bucketed by their prediction-time `p_known` and the
accuracy of the next answer is shown per bucket.

**Rule, fixed before looking:** keep 0.8 / 0.5 unless another pair separates the buckets clearly better. "Separates"
means the smaller of the two gaps (strong minus shaky accuracy, shaky minus weak accuracy), and a pair only counts
when every bucket holds at least 5% of the answers. Another pair replaces 0.8 / 0.5 only if its smaller gap is larger
by at least 0.03 (three accuracy points).

In [4]:
candidates = [(0.7, 0.4), (0.8, 0.5), (0.9, 0.6)]
table = []
for hi, lo in candidates:
    band = np.where(pred.p_known >= hi, 'strong', np.where(pred.p_known >= lo, 'shaky', 'weak'))
    g = pred.groupby(band).correct.agg(['mean', 'size'])
    acc = {b: g['mean'].get(b, np.nan) for b in ('strong', 'shaky', 'weak')}
    share = {b: g['size'].get(b, 0) / len(pred) for b in ('strong', 'shaky', 'weak')}
    table.append({'strong >=': hi, 'shaky >=': lo,
                  'acc strong': acc['strong'], 'acc shaky': acc['shaky'], 'acc weak': acc['weak'],
                  'share strong': share['strong'], 'share shaky': share['shaky'], 'share weak': share['weak'],
                  'min gap': min(acc['strong'] - acc['shaky'], acc['shaky'] - acc['weak']),
                  'min share': min(share.values())})
table = pd.DataFrame(table).round(3)
table

,strong >=,shaky >=,acc strong,acc shaky,acc weak,share strong,share shaky,share weak,min gap,min share
0,0.7,0.4,0.741,0.583,0.363,0.728,0.118,0.155,0.158,0.118
1,0.8,0.5,0.788,0.634,0.423,0.505,0.269,0.226,0.154,0.226
2,0.9,0.6,0.796,0.648,0.446,0.457,0.287,0.256,0.148,0.256


In [5]:
ok = table[table['min share'] >= 0.05]
default = table[(table['strong >='] == 0.8) & (table['shaky >='] == 0.5)].iloc[0]
best = ok.loc[ok['min gap'].idxmax()]
if best['min gap'] - default['min gap'] >= 0.03:
    chosen = best
else:
    chosen = default
thresholds = {'strong': float(chosen['strong >=']), 'shaky': float(chosen['shaky >='])}
print(f"default 0.8/0.5 min gap {default['min gap']:.3f}; best eligible {best['strong >=']}/{best['shaky >=']} "
      f"min gap {best['min gap']:.3f}")
print('chosen:', thresholds)

default 0.8/0.5 min gap 0.154; best eligible 0.7/0.4 min gap 0.158
chosen: {'strong': 0.8, 'shaky': 0.5}


In [6]:
bkt.save(params, thresholds, auc=round(float(auc), 4), baseline_auc=round(float(baseline_auc), 4),
         source='ASSISTments 2009, notebooks/10_knowledge_map.ipynb')
print(bkt.PATH.read_text())
assert bkt.load() == (params, thresholds)

{
 "params": {
  "init": 0.7406692765246253,
  "learn": 0.05502086979119816,
  "slip": 0.20440710734048892,
  "guess_choice": 0.5,
  "guess_written": 0.26175108970449185
 },
 "thresholds": {
  "strong": 0.8,
  "shaky": 0.5
 },
 "auc": 0.6899,
 "baseline_auc": 0.695,
 "source": "ASSISTments 2009, notebooks/10_knowledge_map.ipynb"
}



## Results and limits

- The figures above are on students the model never saw. **BKT with one shared parameter set does not beat the
  tally**: AUC 0.690 against 0.695 for the running accuracy of the same student on the same skill, and the tally is
  ahead on choice and written answers alike. Different starting values reach the same fit, so this is not a failed
  optimisation. The tally knows how strong *this* student is; the shared BKT parameters only know the average
  student, and 0.005 AUC is what that costs here. The two are close enough that BKT is kept for what the tally cannot
  give: a state that moves with each answer, a learning step, and a different weight for choice and written answers.
  It is not kept because it predicts better.
- `guess_choice` sits on its upper bound (0.5): the data would take a higher value, so the choice guess is a ceiling
  here, not an estimate. The fitted `init` (0.74) is high because most students already know the skill when they
  reach it in a skill builder.
- The data are **middle-school maths skills** answered in a tutoring system, not university subjects. The parameters
  are a prior that is reasonable for "a student answering questions on a concept"; they are not a claim about any
  particular course. Per-concept parameters are not fitted: StudyFlow concepts have too few answers each.
- The three bands are bucket labels for the map. The rule above picks the cut-offs from how well the buckets
  separate next-answer accuracy, nothing more.
- That the knowledge map shows the **weaker** of its two signals (BKT understanding, memory-forecast retention) is a
  design choice made for the product, not something fitted here.